In [0]:
%run ../utils/adls_auth

In [0]:
%run ../utils/control_table

In [0]:
# Disable deletion vectors for Synapse compatibility
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import col, sha2, concat_ws, date_trunc, unix_timestamp
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit,hour
import uuid


spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

TRIPS_SILVER = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_silver"
GOLD_PATH = "abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips"

dim_date = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_date")
dim_location = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_location")
dim_vendor = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_vendor")
dim_weather = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_weather")
batch_id = str(uuid.uuid4())


In [0]:

trips_df = spark.read.format("delta").load(TRIPS_SILVER)

fact_df = (
    trips_df
    .join(dim_date, trips_df.pickup_date == dim_date.full_date, "left")
    .join(
        dim_location.alias("pu"),
        (trips_df.PULocationID == col("pu.location_id")) &
        (trips_df.pickup_date >= col("pu.effective_start_date")) &
        (trips_df.pickup_date <= col("pu.effective_end_date")),
        "left",
    )
    .join(
        dim_location.alias("do"),
        (trips_df.DOLocationID == col("do.location_id")) &
        (trips_df.pickup_date >= col("do.effective_start_date")) &
        (trips_df.pickup_date <= col("do.effective_end_date")),
        "left",
    )
    .join(dim_vendor, trips_df.VendorID == dim_vendor.vendor_id, "left")
    .join(
        dim_weather,
        date_trunc("hour", trips_df.tpep_pickup_datetime) == dim_weather.weather_hour,
        "left",
    )
    .select(
        sha2(concat_ws("_", trips_df.VendorID, trips_df.tpep_pickup_datetime,
                       trips_df.tpep_dropoff_datetime, trips_df.PULocationID), 256).alias("trip_key"),
        dim_date.date_key.alias("pickup_date_key"),
        col("pu.location_key").alias("pickup_location_key"),
        col("do.location_key").alias("dropoff_location_key"),
        dim_vendor.vendor_key.alias("vendor_key"),
        dim_weather.weather_key.alias("weather_key"),
        hour(col("tpep_pickup_datetime")).alias("pickup_hour"),
        trips_df.trip_distance,
        trips_df.fare_amount,
        trips_df.tip_amount,
        trips_df.total_amount,
        trips_df.passenger_count,
        (unix_timestamp(trips_df.tpep_dropoff_datetime) - unix_timestamp(trips_df.tpep_pickup_datetime)).alias("trip_duration_seconds"),
    )
    .withColumn("_batch_id", lit(batch_id))
    .withColumn("_created_at", current_timestamp())
    .withColumn("_updated_at", current_timestamp())
)


In [0]:
if not DeltaTable.isDeltaTable(spark, GOLD_PATH):
    fact_df.write.format("delta").mode("overwrite").partitionBy("pickup_date_key").save(GOLD_PATH)
else:
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
    fact_table = DeltaTable.forPath(spark, GOLD_PATH)
    
    
    update_cols = [c for c in fact_df.columns if c not in ("_created_at", "_batch_id")]
    update_map = {c: col(f"source.{c}") for c in update_cols}
    update_map["_updated_at"] = current_timestamp()
    
    (fact_table.alias("target")
        .merge(fact_df.alias("source"), "target.trip_key = source.trip_key")
        .whenMatchedUpdate(set=update_map)
        .whenNotMatchedInsertAll()
        .execute())
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")

print(f"fact_trips built/merged: {fact_df.count()} rows.")